# AB-SAWの4分子・6分子閉路

Google Driveの既存試料を読み、C₄・C₆の全体、最大成分内、最大成分外の個数を集計します。分子列が同じ閉路は回転・逆向きを同一とし、同じ分子ペアの並列結合を1本にまとめます。

**初回は下のセルを上から実行し、Driveへのアクセスを許可してください。** 初期設定の `smoke` は保存済み1試料を読みます。成功すると `L128_phi0.10_i000000` の全体で C₄=6、C₆=0 になることを表示します。

| MODE | 対象 |
|---|---|
| `smoke` | 実試料1件。入出力と集計の確認 |
| `plan` | 条件と件数を検査。元の結合ファイルを開かない |
| `pilot` | L=32,64,128、目標占有率0.10〜0.70を0.01刻み、各条件1試料 |
| `full` | 同じ183条件で各100試料、合計18,300試料 |

`pilot` と `full` は同一出力先を使用し、入力・コードが一致する計算済み試料を再利用します。同じ出力先で同時に実行する場合は、先の実行が終了するのを待ってください。実験は占有率ごとに独立生成した保存済み試料を使用します。


In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

MODE = "smoke"  # smoke / plan / pilot / full
DRIVE_ROOT = Path("/content/drive/MyDrive/tetra-peg-network")
STUDY = DRIVE_ROOT / "backup" / "h2_relation_fine_compact"
ARCHIVE = DRIVE_ROOT / "backup" / "h2_archive_20261028"
SAMPLES = STUDY / "results_fe428bde2d" / "samples.csv"
OUTPUT_ROOT = DRIVE_ROOT / "cycle46_report" / "all_occupancies"


In [ ]:
import csv
from pathlib import Path
import subprocess
import sys
import tempfile
from IPython.display import Image, display

if MODE not in {"smoke", "plan", "pilot", "full"}:
    raise ValueError("MODE は smoke / plan / pilot / full から選んでください。")

sample_candidates = [SAMPLES, SAMPLES.with_name("upload_h2_validation.zip")]
samples = next((path for path in sample_candidates if path.is_file()), None)
if samples is None:
    raise FileNotFoundError(f"試料表がありません。SAMPLES を samples.csv または保存済みZIPに設定してください。確認先 {SAMPLES}")

case = "L128_phi0.10_i000000"
quick_run = STUDY / "cases" / case / "runs" / case
has_loose_case = all(
    (quick_run / name).is_file() or (quick_run / (name + ".gz")).is_file()
    for name in ("bonds.csv", "cluster-members.csv")
)
if MODE == "smoke" and has_loose_case:
    source = ["--study", str(STUDY)]
elif (ARCHIVE / "h2.tar.part-000").is_file() or MODE == "plan":
    source = ["--archive", str(ARCHIVE)]
    if MODE == "smoke":
        print("分割バックアップで対象試料まで順に読みます。")
else:
    source = ["--study", str(STUDY)]

branch = "codex/ab-saw-cycle46-colab"
with tempfile.TemporaryDirectory(prefix="cycle46_repo_") as temp:
    repo = Path(temp) / "tetra-peg-network"
    subprocess.run([
        "git", "clone", "--quiet", "--depth", "1", "--branch", branch,
        "https://github.com/takakei510/tetra-peg-network.git", str(repo),
    ], check=True)
    commit = subprocess.check_output(["git", "-C", str(repo), "rev-parse", "HEAD"], text=True).strip()
    print(f"GitHub commit: {commit}", flush=True)
    output = OUTPUT_ROOT / ("smoke" if MODE == "smoke" else "comparison")
    cmd = [
        sys.executable, str(repo / "scripts" / "analysis" / "compare_cycles_46.py"),
        "--repo", str(repo), "--samples", str(samples), *source,
        "--output", str(output), "--phis", "all",
    ]
    if MODE == "plan":
        cmd += ["--plan"]
    elif MODE == "smoke":
        cmd += ["--replicates", "1", "--case", case]
    elif MODE == "pilot":
        cmd += ["--replicates", "1"]
    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    try:
        for line in process.stdout:
            print(line, end="", flush=True)
        if process.wait() != 0:
            raise subprocess.CalledProcessError(process.returncode, cmd)
    finally:
        if process.poll() is None:
            process.terminate()
            process.wait()

if MODE == "smoke":
    with (output / "cycle46_samples.csv").open(encoding="utf-8-sig", newline="") as handle:
        row = next(csv.DictReader(handle))
    actual = (int(row["C4_mol"]), int(row["C6_mol"]))
    if row["case_id"] != case or actual != (6, 0):
        raise ValueError(f"保存済み試料の参照値と一致しません。{row['case_id']} C4,C6 = {actual}")
    print(f"{case}: C4={actual[0]}, C6={actual[1]}。参照値と一致しました。")
elif MODE in {"pilot", "full"}:
    for name in ("01_occupancy.png", "02_largest_component.png", "03_lattice_size.png"):
        display(Image(filename=str(output / name)))

if MODE != "plan":
    print(f"出力先 {output}")
